<a href="https://colab.research.google.com/github/astronomertypes/Stardew-Valley-QnA/blob/main/StardewValleyRaG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 — Install + imports

In [1]:
!pip install -q sentence-transformers faiss-cpu requests beautifulsoup4 google-genai

import requests
import re
import time
import pickle
import numpy as np
import faiss
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 63.3 MB/s eta 0:00:00


Cell 2 — Wiki API config + categories (includes Locations fix)

In [2]:
API_URL = "https://stardewvalleywiki.com/mediawiki/api.php"
HEADERS = {
    'User-Agent': 'StardewRAGBot/1.0 (personal project; contact: youremail@example.com)'
}

CATEGORIES = [
    "Category:NPCs",
    "Category:Crops",
    "Category:Fish",
    "Category:Animals",
    "Category:Artisan Goods",
    "Category:Festivals",
    "Category:Locations",   # needed for pages like "The Mines"
]

def get_category_contents(category):
    pages, subcats = [], []
    cmcontinue = None
    while True:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": category,
            "cmlimit": 500,
            "cmtype": "page|subcat",
            "format": "json",
        }
        if cmcontinue:
            params["cmcontinue"] = cmcontinue
        r = requests.get(API_URL, params=params, headers=HEADERS, timeout=30).json()
        if "error" in r:
            print(f"API Error on {category}: {r['error']}")
            break
        for m in r.get("query", {}).get("categorymembers", []):
            if m["title"].startswith("Category:"):
                subcats.append(m["title"])
            else:
                pages.append(m["title"])
        cmcontinue = r.get("continue", {}).get("cmcontinue")
        if not cmcontinue:
            break
    return pages, subcats

def get_pages_in_category_recursive(category, max_depth=2):
    visited_cats = set()
    all_pages = set()
    def walk(cat, depth):
        if cat in visited_cats or depth > max_depth:
            return
        visited_cats.add(cat)
        pages, subcats = get_category_contents(cat)
        all_pages.update(pages)
        for sub in subcats:
            walk(sub, depth + 1)
    walk(category, 0)
    return sorted(all_pages)

all_titles = set()
for cat in CATEGORIES:
    try:
        found = get_pages_in_category_recursive(cat)
        print(f"{cat}: {len(found)} pages found (incl. subcategories)")
        all_titles.update(found)
    except Exception as e:
        print(f"Failed on {cat}: {e}")

# Safety net for a few key villagers + explicit Mines pages
villager_titles = ["Abigail", "Sebastian", "Shane", "Penny", "Harvey", "Leah",
                    "Elliott", "Maru", "Sam", "Vincent", "Jas"]
all_titles.update(villager_titles)
all_titles.update(["The Mines", "Skull Cavern"])

all_titles = sorted(all_titles)
print(f"\nTotal unique pages to fetch: {len(all_titles)}")

Category:NPCs: 50 pages found (incl. subcategories)
Category:Crops: 147 pages found (incl. subcategories)
Category:Fish: 72 pages found (incl. subcategories)
Category:Animals: 32 pages found (incl. subcategories)
Category:Artisan Goods: 36 pages found (incl. subcategories)
Category:Festivals: 13 pages found (incl. subcategories)
Category:Locations: 85 pages found (incl. subcategories)

Total unique pages to fetch: 433


Cell 3 — Fetch + clean (ONE version only — HTML/BeautifulSoup based)

In [3]:
def get_page_text(title):
    """Fetch rendered HTML (so templates like gift-tables are expanded) and convert to clean text."""
    params = {
        "action": "parse",
        "page": title,
        "prop": "text",
        "format": "json",
        "redirects": 1,
    }
    r = requests.get(API_URL, params=params, headers=HEADERS, timeout=30).json()
    if "error" in r:
        return ""
    html = r.get("parse", {}).get("text", {}).get("*", "")
    if not html:
        return ""
    return clean_html(html)


def clean_html(html):
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup.find_all(["style", "script", "sup", "table"]):
        if tag.name == "table" and any(
            k in tag.get_text().lower() for k in ["love", "like", "dislike", "hate", "neutral"]
        ):
            continue
        tag.decompose()

    for table in soup.find_all("table"):
        rows = []
        for tr in table.find_all("tr"):
            cells = [c.get_text(separator=", ", strip=True) for c in tr.find_all(["td", "th"])]
            if cells:
                rows.append(": ".join(cells))
        table.replace_with("\n".join(rows))

    text = soup.get_text(separator="\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

Cell 4 — Fetch all pages into raw_docs

In [4]:
raw_docs = []

for i, title in enumerate(all_titles):
    try:
        text = get_page_text(title)
        if not text:
            print(f"Warning: Page '{title}' returned empty text.")
    except requests.exceptions.JSONDecodeError as e:
        print(f"Warning: Could not decode JSON for page '{title}': {e}")
        text = ""
    except requests.exceptions.ReadTimeout as e:
        print(f"Warning: Read timeout for page '{title}': {e}")
        text = ""
    except Exception as e:
        print(f"Warning: An unexpected error occurred for page '{title}': {e}")
        text = ""

    if text and len(text.strip()) > 5:
        raw_docs.append({"title": title, "text": text.strip()})

    if i % 10 == 0:
        print(f"{i}/{len(all_titles)} processed...")
    time.sleep(0.1)

print(f"\nFetched {len(raw_docs)} non-empty pages.")

with open("raw_docs.pkl", "wb") as f:
    pickle.dump(raw_docs, f)

0/433 processed...
10/433 processed...
20/433 processed...
30/433 processed...
40/433 processed...
50/433 processed...
60/433 processed...
70/433 processed...
80/433 processed...
90/433 processed...
100/433 processed...
110/433 processed...
120/433 processed...
130/433 processed...
140/433 processed...
150/433 processed...
160/433 processed...
170/433 processed...
180/433 processed...
190/433 processed...
200/433 processed...
210/433 processed...
220/433 processed...
230/433 processed...
240/433 processed...
250/433 processed...
260/433 processed...
270/433 processed...
280/433 processed...
290/433 processed...
300/433 processed...
310/433 processed...
320/433 processed...
330/433 processed...
340/433 processed...
350/433 processed...
360/433 processed...
370/433 processed...
380/433 processed...
390/433 processed...
400/433 processed...
410/433 processed...
420/433 processed...
430/433 processed...

Fetched 433 non-empty pages.


Cell 5 — Chunking (prose chunks with overlap) + gift-preference chunks (ONE version each)

In [5]:
def chunk_text(title, text, max_chars=600, overlap=100):
    paragraphs = re.split(r"\n\s*\n", text)

    def split_long_paragraph(para, limit):
        if len(para) <= limit:
            return [para]
        sentences = re.split(r"(?<=[.!?])\s+", para)
        pieces, cur = [], ""
        for s in sentences:
            if len(cur) + len(s) < limit:
                cur += " " + s
            else:
                if cur.strip():
                    pieces.append(cur.strip())
                cur = s
        if cur.strip():
            pieces.append(cur.strip())
        return pieces

    normalized_paragraphs = []
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        normalized_paragraphs.extend(split_long_paragraph(para, max_chars))

    chunks = []
    current = ""
    for para in normalized_paragraphs:
        if len(current) + len(para) < max_chars:
            current += (" " + para if current else para)
        else:
            if current.strip():
                chunks.append(current.strip())
            tail = current[-overlap:] if overlap > 0 else ""
            current = (tail + " " + para).strip() if tail else para
    if current.strip():
        chunks.append(current.strip())

    return [{"title": title, "text": f"{title}: {c}"} for c in chunks if len(c) > 30]


GIFT_LABELS = ["Loved Gifts", "Liked Gifts", "Neutral Gifts", "Disliked Gifts", "Hated Gifts"]
_label_alt = "|".join(GIFT_LABELS)
LABEL_PATTERN = re.compile(
    rf"({_label_alt}):\s*(.+?)(?=(?:{_label_alt}):|Contents|\n\n|\Z)",
    re.DOTALL
)
VERB_MAP = {
    "Loved Gifts": "loves",
    "Liked Gifts": "likes",
    "Neutral Gifts": "is neutral about",
    "Disliked Gifts": "dislikes",
    "Hated Gifts": "hates",
}

def make_gift_chunks(title, text):
    chunks = []
    for label, items in LABEL_PATTERN.findall(text):
        items = items.strip().rstrip(",. ")
        if not items or len(items) > 300:
            continue
        sentence = f"{title} {VERB_MAP[label]} these gifts: {items}."
        chunks.append({"title": title, "text": sentence})
    return chunks

Cell 6 — Build all_chunks (ONE version, combines both chunk types)

In [14]:
import re

GIFT_LABELS = ["Loved Gifts", "Liked Gifts", "Neutral Gifts", "Disliked Gifts", "Hated Gifts"]
_label_alt = "|".join(GIFT_LABELS)
LABEL_PATTERN = re.compile(
    rf"({_label_alt}):\s*(.+?)(?=(?:{_label_alt}):|Contents|\n\n|\Z)",
    re.DOTALL
)

VERB_MAP = {
    "Loved Gifts": "loves",
    "Liked Gifts": "likes",
    "Neutral Gifts": "is neutral about",
    "Disliked Gifts": "dislikes",
    "Hated Gifts": "hates",
}

def make_gift_chunks(title, text):
    chunks = []
    for label, items in LABEL_PATTERN.findall(text):
        items = items.strip().rstrip(",. ")
        if not items or len(items) > 300:  # sanity guard against a bad match swallowing too much
            continue
        sentence = f"{title} {VERB_MAP[label]} these gifts: {items}."
        chunks.append({"title": title, "text": sentence})
    return chunks

In [15]:
all_chunks = []
for doc in raw_docs:
    all_chunks.extend(chunk_text(doc["title"], doc["text"]))
    all_chunks.extend(make_gift_chunks(doc["title"], doc["text"]))

print(f"Total chunks: {len(all_chunks)}")

if len(all_chunks) > 0:
    print(all_chunks[0])
    lengths = [len(c["text"]) for c in all_chunks]
    print(f"Avg chunk length: {sum(lengths)/len(lengths):.0f} chars "
          f"(min={min(lengths)}, max={max(lengths)})")
else:
    print("Warning: No chunks created. Check raw_docs.")

Total chunks: 4147
{'title': '1 River Road', 'text': "1 River Road: 1 River Road\n is the home of \nGeorge\n, \nEvelyn\n, and \nAlex\n. It's just southeast of \nPierre's General Store\n and just behind \nthe saloon\n. Interior\n[\nedit\n]"}
Avg chunk length: 529 chars (min=27, max=7701)


In [16]:
for c in all_chunks:
    if c["title"] == "Abigail" and "loves these gifts" in c["text"]:
        print(c["text"])

Abigail loves these gifts: Amethyst, Banana Pudding, Blackberry Cobbler, Chocolate Cake, Monster Compendium, Pufferfish, Pumpkin, Spicy Eel.


Cell 7 — Embed + build FAISS index

In [17]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in all_chunks]
if len(texts) == 0:
    raise ValueError("all_chunks is empty — nothing to embed.")

embeddings = embedder.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=64,
)
embeddings = np.array(embeddings).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"Indexed {index.ntotal} chunks, dim={dim}")
assert index.ntotal == len(all_chunks), "Mismatch between index size and chunk metadata!"

faiss.write_index(index, "stardew.index")
with open("chunks_meta.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

print("Saved stardew.index and chunks_meta.pkl")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/65 [00:00<?, ?it/s]

Indexed 4147 chunks, dim=384
Saved stardew.index and chunks_meta.pkl


Cell 8 — Retrieval function (ONE version)

In [18]:
def retrieve(query, k=6):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        results.append({"score": float(score), **all_chunks[idx]})
    return results

Cell 9 — Gemini client + generation (new SDK, thinking-tokens fix)

In [19]:
from google import genai
from google.genai import types
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.6-flash"

SYSTEM_PROMPT = """You are a Stardew Valley expert assistant.
Answer the user's question using ONLY the CONTEXT provided below.
Rules:
- Respond in exactly ONE short sentence. No preamble, no lists, no extra detail.
- Do not say "based on the context" or similar — just answer directly.
- If the context does not contain the answer, respond with exactly: "I don't know based on the wiki data I have."
- Do not use outside knowledge beyond the context.
"""
FALLBACK_MESSAGE = "I don't know based on the wiki data I have."


def generate_answer(query, context_chunks):
    context = "\n---\n".join(c["text"] for c in context_chunks)
    prompt = f"""{SYSTEM_PROMPT}

CONTEXT:
{context}

QUESTION: {query}

ONE-LINE ANSWER:"""

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1,
                max_output_tokens=200,
                thinking_config=types.ThinkingConfig(thinking_level="minimal"),
            ),
        )
    except TypeError:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1,
                max_output_tokens=200,
                thinking_config=types.ThinkingConfig(thinking_budget=0),
            ),
        )

    try:
        generated_text = response.text.strip()
        return generated_text if generated_text else FALLBACK_MESSAGE
    except (ValueError, AttributeError) as e:
        print(f"DEBUG: issue getting response text: {e}")
        return FALLBACK_MESSAGE

Cell 10 — Ask pipeline + test

In [23]:
import time as _time

def ask(query, k=6, verbose=False):
    chunks = retrieve(query, k=k)
    if verbose:
        print("Retrieved chunks:")
        for c in chunks:
            print(f"  [{c['score']:.3f}] {c['text'][:80]}...")
        print()
    return generate_answer(query, chunks)

questions = [
    "What are Abigail's loved gifts?",
    "How do I get to the mines?",
    "What day is the Flower Dance?",
    "Can you marry Linus?",
]

print("Running questions...\n")
for query in questions:
    answer = ask(query)
    print(f"Q: {query}")
    print(f"A: {answer}")
    print("-" * 40)
    _time.sleep(4)

Running questions...



ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 41.379399277s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},

Cell 11 — Gradio demo UI

In [ ]:
import gradio as gr

def gradio_ask(query):
    if not query.strip():
        return ""
    return ask(query)

demo = gr.Interface(
    fn=gradio_ask,
    inputs=gr.Textbox(label="Ask about Stardew Valley", placeholder="What does Penny like?"),
    outputs=gr.Textbox(label="Answer"),
    title="Stardew Valley RAG Assistant",
    description="Answers are grounded in the Stardew Valley Wiki via retrieval-augmented generation.",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f084c6a5b02a392c1a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
